# Hands-On Lab: Pandas Data Wrangling for Machine Learning
Accompanies **Appendix B: Pandas Data Wrangling for Machine Learning**.

In this lab, we:
1. Downcast data types to slash memory consumption by 75%.
2. Audit missing values and apply leak-free training imputation.
3. Feature engineering via the **Split-Apply-Combine** (`groupby`) pattern.
4. Align one-hot dummy schemas between training and testing sets.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_rows = 10_000

df = pd.DataFrame({
    'CustomerID': np.random.randint(1000, 1500, size=n_rows),
    'State': np.random.choice(['CA', 'NY', 'TX', 'FL', 'IL'], size=n_rows),
    'Amount': [f"${x:,.2f}" if np.random.rand() > 0.05 else "MISSING" for x in np.random.exponential(scale=1000, size=n_rows)],
    'Score': np.random.randint(300, 850, size=n_rows).astype(np.int64)
})

print(f"Raw DataFrame Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Clean & downcast
df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').astype(np.float32)
df['State'] = df['State'].astype('category')
df['Score'] = df['Score'].astype(np.int16)

print(f"Optimized DataFrame Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Groupby Aggregations
summary = df.groupby('CustomerID').agg(
    total_spend=('Amount', 'sum'),
    avg_spend=('Amount', 'mean'),
    tx_count=('Amount', 'count')
).reset_index()

print("\nEngineered Customer Summary Table:")
print(summary.head())